# A11 - Bootstrapping + Aggregating

Cecilia Quiroz Hidalgo

### Regresión con OLS

In [24]:
import pandas as pd 
import statsmodels.api as sm

In [25]:
df = pd.read_csv('Advertising.csv')

In [26]:
X = df[["TV", "radio", "newspaper"]]
y = df["sales"]

In [27]:
X = sm.add_constant(X)
modelo = sm.OLS(y, X).fit()
print(modelo.summary())

                            OLS Regression Results                            
Dep. Variable:                  sales   R-squared:                       0.897
Model:                            OLS   Adj. R-squared:                  0.896
Method:                 Least Squares   F-statistic:                     570.3
Date:                Thu, 19 Nov 2026   Prob (F-statistic):           1.58e-96
Time:                        17:10:10   Log-Likelihood:                -386.18
No. Observations:                 200   AIC:                             780.4
Df Residuals:                     196   BIC:                             793.6
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          2.9389      0.312      9.422      0.0

In [28]:
X = X.drop(columns="const")

### Bootstrapping

In [29]:
from sklearn.utils import resample

In [30]:
resample?

Signature:
resample(
    *arrays,
    replace=True,
    n_samples=None,
    random_state=None,
    stratify=None,
    sample_weight=None,
)
Docstring:
Resample arrays or sparse matrices in a consistent way.

The default strategy implements one step of the bootstrapping
procedure.

Parameters
----------
*arrays : sequence of array-like of shape (n_samples,) or             (n_samples, n_outputs)
    Indexable data-structures can be arrays, lists, dataframes or scipy
    sparse matrices with consistent first dimension.

replace : bool, default=True
    Implements resampling with replacement. It must be set to True
    whenever sampling with non-uniform weights: a few data points with very large
    weights are expected to be sampled several times with probability to preserve
    the distribution induced by the weights. If False, this will implement
    (sliced) random permutations.

n_samples : int, default=None
    Number of samples to generate. If left to None this is
    automatically 

Signature:
resample(
    *arrays,
    replace=True,
    n_samples=None,
    random_state=None,
    stratify=None,
    sample_weight=None,
)
Docstring:
Resample arrays or sparse matrices in a consistent way.

The default strategy implements one step of the bootstrapping
procedure.

Parameters
----------
*arrays : sequence of array-like of shape (n_samples,) or             (n_samples, n_outputs)
    Indexable data-structures can be arrays, lists, dataframes or scipy
    sparse matrices with consistent first dimension.

replace : bool, default=True
    Implements resampling with replacement. It must be set to True
    whenever sampling with non-uniform weights: a few data points with very large
    weights are expected to be sampled several times with probability to preserve
    the distribution induced by the weights. If False, this will implement
    (sliced) random permutations.

n_samples : int, default=None
    Number of samples to generate. If left to None this is
    automatically 

In [31]:
Xb, yb = resample(X, y, random_state=5)

Regresión con LinearRegression

In [32]:
from sklearn.linear_model import LinearRegression

In [33]:
modelo = LinearRegression()
modelo.fit(Xb, yb)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [34]:
modelo.intercept_

np.float64(2.534563024735448)

In [35]:
modelo.coef_

array([0.04675824, 0.18447237, 0.00713704])

Con semilla 6: 

In [36]:
Xb, yb = resample(X, y, random_state=6)

In [37]:
modelo = LinearRegression()
modelo.fit(Xb, yb)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [38]:
modelo.intercept_

np.float64(3.2669989017598606)

In [39]:
modelo.coef_

array([ 0.04531755,  0.18137   , -0.00201614])

Con semilla 7:

In [40]:
Xb, yb = resample(X, y, random_state=7)

In [41]:
modelo = LinearRegression()
modelo.fit(Xb, yb)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [42]:
modelo.intercept_

np.float64(3.1393730711080288)

In [43]:
modelo.coef_

array([0.04468461, 0.17807546, 0.00355605])

Ciclo for para repetir hasta con semilla 10,000: 

In [45]:
b0 = []
b_tv = []
b_radio = []
b_newspaper = []

for i in range(10000):
    Xb, yb = resample(X, y, random_state=i)
    modelo = LinearRegression().fit(Xb, yb)
    b0.append(modelo.intercept_)
    b_tv.append(modelo.coef_[0])
    b_radio.append(modelo.coef_[1])
    b_newspaper.append(modelo.coef_[2])

Promedio de cada beta: 

In [46]:
print("B0:", sum(b0) / len(b0))
print("TV:", sum(b_tv) / len(b_tv))
print("Radio:", sum(b_radio) / len(b_radio))
print("Newspaper:", sum(b_newspaper) / len(b_newspaper))

B0: 2.9534439182594965
TV: 0.04569540523230202
Radio: 0.18836691153579466
Newspaper: -0.0009026654029186467


Desviación estándar muestral de cada beta: 

In [50]:
import numpy as np

In [52]:
print("B0:", np.std(b0, ddof=1))
print("TV:", np.std(b_tv, ddof=1))
print("Radio:", np.std(b_radio, ddof=1))
print("Newspaper:", np.std(b_newspaper, ddof=1))

B0: 0.3373572111008783
TV: 0.0019086612044736414
Radio: 0.010860014834065238
Newspaper: 0.006480714244211999


### Conclusión

Al sacar los promedios y la desviación estándar de los betas de la regresión lineal con semilla hasta 10,000, pudimos notar que los coeficientes dan casi exactamente lo mismo que haciendo la regresión con OLS, esto nos puede estar indicando que las estimaciones del modelo si son confiables. 